### Function imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
import uxarray as ux
import intake
import numpy as np

import pandas as pd

import healpix as hp
import holoviews as hv

import easygems.healpix as egh
import easygems.remap as egr

import easygems.healpix as eghp

import cmocean
import geoviews.feature as gf


sigma = 5.67E-8  # Stefan-Boltzmann constant (W/m^2/K^4)

### (Optional) Using Dask to parallize tasks

In [2]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from dask.distributed import performance_report

rda_scratch = '/glade/derecho/scratch/khirata/'

cluster = PBSCluster(
    job_name = 'dask-wk24-hpc',
    cores = 8,
    memory = '1024GiB',
    local_directory = rda_scratch+'/dask/spill',
    log_directory = rda_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=8:mem=1024GB',
    queue = 'casper',
    walltime = '02:00:00',
    #interface = 'ib0'
    interface = 'ext'
)
# cluster = PBSCluster(
#     job_name = 'dask-wk24-hpc',
#     cores = 32,
#     memory = '1024GiB',
#     local_directory = rda_scratch+'/dask/spill',
#     log_directory = rda_scratch + '/dask/logs/',
#     resource_spec = 'select=1:ncpus=32:mem=1024GB',
#     queue = 'casper',
#     walltime = '12:00:00',
#     #interface = 'ib0'
#     interface = 'ext'
# )

cluster.scale(2) # large number may result in an error
# cluster.adapt(minimum=2, maximum=32)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.169:38795,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/khirata/hackathon-casper/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Define the catalog

In [3]:
node_id = 'NCAR'
cat = intake.open_catalog("https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml")[node_id]

In [4]:
# list(cat)

# Query a data set
pd.DataFrame(cat['ew_dyamond3_2D'].describe()["user_parameters"])

,name,description,type,allowed,default
0,time,temporal resolution of the dataset,str,"[PT1H, PT3H]",PT1H
1,zoom,zoom resolution of the dataset,int,"[9, 8, 7, 6, 5, 4, 3, 2, 1]",7


### Define the zoom level

In [5]:
time_slice = slice("2019-01-01", "2021-03-01")

In [6]:
# zoom_lev = 7
zoom_lev = 8

In [7]:
#tmp = cat['ew_dyamond3_2D'](zoom=zoom_lev,time='PT1H').to_dask()
#tmp

### Read in data into UxArray (precipitation)

In [36]:
if node_id=='NERSC':
    scl_icon = 86400. #kg/m2/s to mm/day
    ds_icon = cat['icon_d3hp003'](zoom=zoom_lev).to_dask()
    uxds_icon = ux.UxDataset.from_healpix(ds_icon)
    uxda_pr_icon = uxds_icon['pr'].sel(time=time_slice) #*scl_icon # mm/day

    scl_scream = 86400.*1000. #m/s to mm/day
    ds_scream = cat['scream2D_hrly'](zoom=zoom_lev).to_dask()
    uxds_scream = ux.UxDataset.from_healpix(ds_scream)
    uxda_pr_scream = uxds_scream['pr'].sel(time=time_slice)# *scl_scream # mm/day

    scl_nicam = 86400. #kg/m2/s to mm/day
    ds_nicam = cat['nicam_gl11'](zoom=zoom_lev, time='PT3H').to_dask()
    uxds_nicam = ux.UxDataset.from_healpix(ds_nicam)
    uxds_nicam.assign_coords(hour=uxds_nicam.time.dt.hour)
    uxds_nicam.assign_coords(month=uxds_nicam.time.dt.month)
    uxda_pr_nicam = uxds_nicam['pr'].sel(time=time_slice) #*scl_nicam # mm/day

    scl_um = 86400. #m/s to mm/day
    pth='/global/cfs/cdirs/m4581/gsharing/hackathon/UM/glm.n2560_RAL3p3/'
    fn=pth+'data.healpix.PT1H.z'+str(zoom_lev)+'.zarr'
    ds_um=xr.open_dataset(fn)
    uxds_um = ux.UxDataset.from_healpix(ds_um)
    uxds_um.assign_coords(hour=uxds_um.time.dt.hour)
    uxds_um.assign_coords(month=uxds_um.time.dt.month)
    uxda_pr_um = uxds_um['pr'].sel(time=time_slice) #*scl_um # mm/day

    scl_cas = 24*1000. #assume m/hr
    ds_cas = cat['casesm2_10km_nocumulus'](zoom=zoom_lev,time='PT1H').to_dask()
    uxds_cas = ux.UxDataset.from_healpix(ds_cas)
    uxds_cas.assign_coords(hour=uxds_cas.time.dt.hour)
    uxds_cas.assign_coords(month=uxds_cas.time.dt.month)
    uxda_pr_cas = uxds_cas['pr'].sel(time=time_slice) #*scl_cas # mm/day

#scl_mpas = 48. #mm/30min to mm/day
#ds_mpas = cat['mpas_dyamond3'](zoom=zoom_lev, time='PT30M').to_dask()
#uxds_mpas = ux.UxDataset.from_healpix(ds_mpas)
#uxda_pr_mpas = uxds_mpas['rainnc'].sel(time=time_slice) #*scl_nicam # mm/day

# scl_imerg = 24. #mm/hr to mm/day
# uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/andrew/hackathon/IMERG_V07B_hp9.zarr')
# scl_imerg = 24. #mm/hr to mm/day
# ds_imerg = cat['IR_IMERG'](zoom=9).to_dask()
# uxds_imerg = ux.UxDataset.from_healpix(ds_imerg)

# uxda_pr_imerg_fine = uxds_imerg['precipitation'].sel(time=time_slice)
# uxda_pr_imerg_fine = uxda_pr_imerg_fine.chunk({'time': 48, 'n_face': uxda_pr_imerg_fine.sizes['n_face']})

# level_down = 9 - zoom_lev
# uxda_pr_imerg_tmp = uxda_pr_imerg_fine
# for i in range(level_down):
#     uxda_pr_imerg_tmp = uxda_pr_imerg_tmp.coarsen(n_face=4).mean()
#     uxda_pr_imerg_tmp['crs'].attrs['healpix_nside'] = 2**int(9 - i - 1)
# uxda_pr_imerg = uxda_pr_imerg_tmp

# # level_down = 9 - zoom_lev
# # for i in range(level_down):
# #     uxds_imerg = uxds_imerg.coarsen(n_face=4).mean()
# #     uxds_imerg['crs'].attrs['healpix_nside'] = 2**int(9 - i - 1) #hp_nside // 2
# # uxda_rlut_imerg_rename = uxds_imerg.rename({'n_face': 'cell'})
# # uxds_imerg = ux.UxDataset.from_healpix(uxda_rlut_imerg_rename)

# # uxda_pr_imerg_orig = uxds_imerg['precipitation'].sel(time=time_slice)
# # uxda_pr_imerg_orig = uxda_pr_imerg_orig.chunk({'time': 48, 'n_face': uxda_pr_imerg_orig.sizes['n_face']})
# uxda_pr_imerg = uxda_pr_imerg.resample(time='1H').first().compute() #*scl_imerg # mm/day

In [7]:
if node_id=='NCAR':
    scl_ew = 86400. #kg/m2/s to mm/day
    ds_ew = cat['ew_dyamond3_2D'](zoom=zoom_lev,time='PT1H').to_dask()
    uxds_ew = ux.UxDataset.from_healpix(ds_ew)
    uxds_ew.assign_coords(hour=uxds_ew.time.dt.hour)
    uxds_ew.assign_coords(month=uxds_ew.time.dt.month)
    uxda_pr_ew = uxds_ew['pr'].sel(time=time_slice) #*scl_ew # mm/day

    scl_mpas = 48. #mm/30min to mm/day [note that MPAS data is stored as cumulative - take diff later]
    ds_mpas = cat['mpas_dyamond3'](zoom=zoom_lev, time='PT30M').to_dask()
    uxds_mpas = ux.UxDataset.from_healpix(ds_mpas)
    uxda_pr_mpas = uxds_mpas['rainnc'].diff(dim="time", label="lower").sel(time=time_slice) #*scl_nicam # mm/day

    scl_scream = 86400.*1000. #m/s to mm/day
    ds_scream = cat['scream2D_hrly'](zoom=zoom_lev).to_dask()
    uxds_scream = ux.UxDataset.from_healpix(ds_scream)
    uxda_pr_scream = uxds_scream['pr'].sel(time=time_slice)# *scl_scream # mm/day

    scl_imerg = 24. #mm/hr to mm/day
    uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/khirata/imerg_zarr/imerg_precipitation_2018_2023_zmlv%d.zarr' % zoom_lev)
    # uxds_imerg = ux.UxDataset.from_healpix('/glade/derecho/scratch/andrew/hackathon/IMERG_V07B_hp9.zarr')
    # ds_imerg = cat['IR_IMERG'](zoom=9).to_dask()
    # uxds_imerg = ux.UxDataset.from_healpix(ds_imerg)
    uxda_pr_imerg = uxds_imerg['precipitation'].sel(time=time_slice) #*scl_imerg # mm/day

    if zoom_lev <= 7:
        scl_era5 = 86400. #kg/m2/s to mm/day
        uxds_era5 = ux.UxDataset.from_healpix('/glade/derecho/scratch/khirata/era5_zarr/era5_mtotpr_2018_2021_zmlv%d.zarr' % zoom_lev)
        uxda_pr_era5 = uxds_era5['mtotpr'].sel(time=time_slice) #*scl_era5 # mm/day


/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),
/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),
/glade/u/apps/opt/conda/envs/2025-digital-earths-global-hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of

#### Check the time period

In [8]:
if node_id=='NCAR':
    print('EarthWorks time:')
    print(uxda_pr_ew.time.min().values)
    print(uxda_pr_ew.time[1].values)
    print(uxda_pr_ew.time.max().values)
    print('MPAS time:')
    print(uxda_pr_mpas.time.min().values)
    print(uxda_pr_mpas.time[1].values)
    print(uxda_pr_mpas.time.max().values)
    print('SCREAM time:')
    print(uxda_pr_scream.time.min().values)
    print(uxda_pr_scream.time[1].values)
    print(uxda_pr_scream.time.max().values)
    print('IMERG time:')
    print(uxda_pr_imerg.time.min().values)
    print(uxda_pr_imerg.time[1].values)
    print(uxda_pr_imerg.time.max().values)
    if zoom_lev <= 7:
        print('ERA5 time:')
        print(uxda_pr_era5.time.min().values)
        print(uxda_pr_era5.time[1].values)
        print(uxda_pr_era5.time.max().values)

EarthWorks time:
2020-03-01T01:00:00.000000000
2020-03-01T02:00:00.000000000
2021-03-01T00:00:00.000000000
MPAS time:
2020-01-20T00:00:00.000000000
2020-01-20T00:30:00.000000000
2020-03-04T23:30:00.000000000
SCREAM time:
2019-08-01 00:00:00
2019-08-01 01:00:00
2020-09-01 00:00:00
IMERG time:
2019-01-01T00:00:00.000000000
2019-01-01T00:30:00.000000000
2021-03-01T23:30:00.000000000


### Test plot the data (zonal mean precip) 
Warning: even at zoom = 8 this blows up memory!

In [22]:
if (zoom_lev < 6):
    if node_id=='NERSC':
        zm_cas=uxda_pr_cas.mean(dim='time').zonal_mean()*scl_cas
        zm_um=uxda_pr_um.mean(dim='time').zonal_mean()*scl_um
        zm_icon=uxda_pr_icon.mean(dim='time').zonal_mean()*scl_icon
        zm_nicam=uxda_pr_nicam.mean(dim='time').zonal_mean()*scl_nicam
        zm_scream=uxda_pr_scream.mean(dim='time').zonal_mean()*scl_scream

        plt.plot(zm_cas.latitudes,zm_cas,label='CAS')
        plt.plot(zm_um.latitudes,zm_um,label='UM')
        plt.plot(zm_icon.latitudes,zm_icon,label='ICON')
        plt.plot(zm_nicam.latitudes,zm_nicam,label='NICAM')
        plt.plot(zm_scream.latitudes,zm_scream,label='SCREAM')
        plt.legend()
    if node_id=='NCAR':
        zm_ew=uxda_pr_ew.isel(time=-4).zonal_mean()*scl_ew
        zm_mpas=uxda_pr_mpas.isel(time=-4).zonal_mean()*scl_mpas
        zm_scream=uxda_pr_scream.isel(time=-4).zonal_mean()*scl_scream
        zm_imerg=uxda_pr_imerg.isel(time=-4).zonal_mean()*scl_imerg
        if zoom_lev <= 7:
            zm_era5=uxda_pr_era5.isel(time=-4).zonal_mean()*scl_era5
        
        plt.plot(zm_ew.latitudes,zm_ew,label='EarthWorks')
        plt.plot(zm_mpas.latitudes,zm_mpas,label='MPAS')
        plt.plot(zm_scream.latitudes,zm_scream,label='SCREAM')
        plt.plot(zm_imerg.latitudes,zm_imerg,label='IMERG')
        if zoom_lev <= 7:
            plt.plot(zm_era5.latitudes,zm_era5,label='ERA5')
        plt.yscale('log')
        plt.legend()
        

### Frequency

In [ ]:
month = 2 # February !!

threshold = 1. # mm/day

if node_id=='NERSC':
    pass
elif node_id=='NCAR':
    frequency_ew = (uxda_pr_ew > threshold / scl_ew).groupby("time.month").mean("time").sel(month=month)
    frequency_mpas = (uxda_pr_mpas > threshold / scl_mpas).groupby("time.month").mean("time").sel(month=month)
    frequency_scream = (uxda_pr_scream > threshold / scl_scream).groupby("time.month").mean("time").sel(month=month)
    frequency_imerg = (uxda_pr_imerg > threshold / scl_imerg).groupby("time.month").mean("time").sel(month=month)
    if zoom_lev <= 7:
        frequency_era5 = (uxda_pr_era5 > threshold / scl_era5).groupby("time.month").mean("time").sel(month=month)


In [ ]:
if node_id=='NERSC':
    pass
elif node_id=='NCAR':
    dir_scratch = '/glade/derecho/scratch/khirata/'
    frequency_ew.to_netcdf(dir_scratch+'pr_freq_ew_z%d_mo%d.nc' % (zoom_lev, month))
    frequency_mpas.to_netcdf(dir_scratch+'pr_freq_mpas_z%d_mo%d.nc' % (zoom_lev, month))
    frequency_scream.to_netcdf(dir_scratch+'pr_freq_scream_z%d_mo%d.nc' % (zoom_lev, month))
    frequency_imerg.to_netcdf(dir_scratch+'pr_freq_imerg_z%d_mo%d.nc' % (zoom_lev, month))
    if zoom_lev <= 7:
        frequency_era5.to_netcdf(dir_scratch+'pr_freq_era5_z%d_mo%d.nc' % (zoom_lev, month))


In [ ]:
cluster.close()

NameError: name 'cluster' is not defined

#### Misc tests

In [25]:
frequency_era5.plot()

:Image   [x,y]   (x_y mtotpr)

In [ ]:
frequency_scream.plot()

:Image   [x,y]   (x_y pr)

In [ ]:
frequency_imerg.plot()

:Image   [x,y]   (x_y precipitation)